# PyMOL RMSD Alignment

![PyMOL RMSD Alignment](https://proto-bio.github.io/proto-assets/images/tool/pymol_rmsd/hero.png)

This notebook demonstrates pairwise structure alignment using Open-Source PyMOL. We align two `Structure` inputs and inspect the RMSD returned by PyMOL `cealign` or regular `align`.

In [1]:
from proto_tools.utils.notebook_docs import display_overview, display_api_reference, display_docs_section, display_doc_link, display_available_tools
display_doc_link("pymol-rmsd")
display_overview("pymol-rmsd")
display_docs_section("pymol-rmsd", "Background")

# PyMOL RMSD

[PyMOL](https://www.pymol.org/) is a molecular visualization system originally written by Warren L. DeLano and maintained as open source by [Schrödinger, LLC](https://www.schrodinger.com/). This toolkit uses Open-Source PyMOL as a headless structural-alignment backend through a single registered tool that aligns two structures and reports the post-alignment RMSD together with method-specific alignment statistics. Two alignment routines are exposed: the Combinatorial Extension `cealign` for distantly related folds and the sequence-aware `align` for closely related structures.

Root mean square deviation (RMSD) measures the average distance between corresponding atoms of two structures after optimal rigid-body superposition. It is the standard summary statistic for assessing how closely a model recapitulates a reference structure, comparing alternative poses of the same complex, and quantifying conformational change between two states of the same protein. Computing a meaningful RMSD requires a residue correspondence between the two structures, which the upstream alignment routine establishes before the superposition is performed.

Open-Source PyMOL exposes two distinct alignment routines that this toolkit invokes. The `cealign` command implements the Combinatorial Extension (CE) algorithm ([Shindyalov and Bourne, 1998](https://doi.org/10.1093/protein/11.9.739)), which builds a structural alignment from aligned fragment pairs based on local geometry rather than a global sequence alignment. CE was developed to detect remote structural similarity below the sequence-similarity twilight zone, where the underlying sequences share too little identity for a meaningful sequence-based alignment. The `align` command in contrast first computes a sequence alignment using the BLOSUM62 substitution matrix, performs a structural superposition over the aligned residues, and then iterates several cycles of outlier rejection to remove residues with poor structural agreement. This routine is appropriate when the two structures share substantial sequence identity and the goal is a residue-matched superposition that excludes locally divergent regions.

## Available tools

In [2]:
display_available_tools("pymol-rmsd")

- **`run_pymol_rmsd_alignment()`** — Pairwise structure RMSD alignment using PyMOL cealign or align.

### `run_pymol_rmsd_alignment`

PyMOL RMSD Alignment performs pairwise structural superposition and returns post-alignment RMSD plus method-specific metrics. The tool accepts standardized `Structure` inputs, so callers can pass raw PDB/CIF content, file paths, or existing `Structure` objects.

In [3]:
from pathlib import Path

from proto_tools import PyMOLRMSDConfig, PyMOLRMSDInput, PyMOLRMSDOutput, Structure, run_pymol_rmsd_alignment

In [4]:
# Display input docs
display_api_reference("pymol-rmsd", "input", "run_pymol_rmsd_alignment")

# Align two homologous TIM-barrel structures (1TIM, 8TIM) fetched from the RCSB,
# so the RMSD reflects a real structural superposition rather than self-alignment.
target = Structure.from_rcsb("1TIM")
mobile = Structure.from_rcsb("8TIM")
inputs = PyMOLRMSDInput(target_structure=target, mobile_structure=mobile)

**Input** — `PyMOLRMSDInput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>target_structure</code> | <code>Structure</code> | required | Target/reference structure (Structure object, file path, or raw PDB/CIF string) |
| <code>mobile_structure</code> | <code>Structure</code> | required | Mobile/query structure (Structure object, file path, or raw PDB/CIF string) |

In [5]:
# Display config docs
display_api_reference("pymol-rmsd", "config", "run_pymol_rmsd_alignment")

config = PyMOLRMSDConfig(method="cealign")

**Config** — `PyMOLRMSDConfig`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>verbose</code> | <code>int</code> | <code>0</code> | Verbosity level (0=quiet, 1=info, 2=debug, 3=raw subprocess stderr). True→1, False→0. |
| <code>device</code> | <code>str</code> | <code>'cpu'</code> | Device to run the tool on (e.g., 'cpu', 'cuda', 'cuda:0', 'cloud') |
| <code>timeout</code> | <code>int &#124; None</code> | <code>3600</code> | Maximum execution time in seconds. None waits indefinitely. |
| <code>seed</code> | <code>int &#124; None</code> | <code>None</code> | Random seed for reproducible results. Some cacheable tools gate cache on this field. |
| <code>method</code> | <code>Literal['cealign', 'align']</code> | <code>'cealign'</code> | PyMOL alignment routine to use for RMSD calculation. |
| <code>target_selection</code> | <code>str</code> | <code>'target'</code> | PyMOL target selection, e.g. 'target and name CA'. |
| <code>mobile_selection</code> | <code>str</code> | <code>'mobile'</code> | PyMOL mobile selection, e.g. 'mobile and name CA'. |
| <code>failure_rmsd</code> | <code>float</code> | <code>999.0</code> | RMSD returned when PyMOL cannot align the requested structures. |

In [6]:
# Run the alignment
result = run_pymol_rmsd_alignment(inputs, config)

Running run_pymol_rmsd_alignment [00:00]

In [7]:
# Display output docs and inspect results
display_api_reference("pymol-rmsd", "output", "run_pymol_rmsd_alignment")

print(f"Method: {result.method}")
print(f"RMSD: {result.rmsd:.6f} Angstrom")
print(dict(result.metrics.items()))

**Output** — `PyMOLRMSDOutput`

| Field | Type | Default | Description |
|-------|------|---------|-------------|
| <code>method</code> | <code>Literal['cealign', 'align']</code> | required | PyMOL alignment method used. |
| <code>metrics</code> | <code>PyMOLRMSDMetrics</code> | <code>PyMOLRMSDMetrics(primary_metric='rmsd', metric_type='PyMOLRMSDMetrics')</code> | RMSD alignment metrics. |

**Metrics** — `PyMOLRMSDMetrics`

| Metric | Type | Range | Unit | Description |
|--------|------|-------|------|-------------|
| <code>rmsd</code> **(primary)** | <code>float</code> | <code>&gt;= 0</code> | <code>angstrom</code> |  |
| <code>aligned_length</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>aligned_atoms</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>alignment_cycles</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>alignment_score</code> | <code>float</code> |  |  |  |
| <code>pre_refinement_rmsd</code> | <code>float</code> | <code>&gt;= 0</code> | <code>angstrom</code> |  |
| <code>pre_refinement_aligned_atoms</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |
| <code>aligned_residues</code> | <code>int</code> | <code>&gt;= 0</code> |  |  |

Method: cealign
RMSD: 0.866855 Angstrom
{'rmsd': 0.8668547656125428, 'aligned_length': 488}


## Export Results

Alignment results can be exported to JSON format for downstream analysis or integration with other computational pipelines.

In [8]:
output_dir = Path("./example_output")
output_dir.mkdir(exist_ok=True)

result.export("pymol_rmsd_result", export_path=output_dir, file_format="json")
print(f"Exported to {output_dir / 'pymol_rmsd_result.json'}")

Exported to example_output/pymol_rmsd_result.json
